This validation is equivalent to the one performed in:

https://github.com/juan11iguel/wct_modelling/blob/dc_model/visualizations/summary_table_and_regression.ipynb

In [1]:
from pathlib import Path
import pandas as pd

from phd_visualizations.regression import regression_plot
from phd_visualizations.calculations import SupportedInstruments 
from phd_visualizations import save_figure

import combined_cooler
import matlab

%reload_ext autoreload
%autoreload 2

data_path = Path("../data")

df = pd.read_csv(data_path / "dc_out_exp.csv")
df


,Tamb,Tin,w_fan,q,Tout
0,22.43,37.54,13.72,5.98,32.95
1,18.90,40.00,11.00,5.83,34.44
2,15.29,42.23,25.00,23.58,38.34
3,15.25,37.55,19.62,17.73,34.00
4,27.86,37.27,17.42,14.91,35.67
5,13.69,37.50,27.95,11.97,30.90
6,25.37,38.90,47.05,12.61,33.06
7,15.10,36.99,11.00,5.31,31.23
8,19.85,40.21,23.90,14.94,36.03
9,13.34,37.43,34.64,23.47,32.85


### Evaluate model

In [5]:
cc_model = combined_cooler.initialize()

results: list[dict] = []
Tdc_out = []
Ce_dc = []
for idx, ds in df.iterrows():

    qdc_m3h = matlab.double([ds["q"]])
    wdc = matlab.double([ds["w_fan"]])
    Tamb_C = matlab.double([ds["Tamb"]])
    Tin_C = matlab.double([ds["Tin"]])

    # Create the 'options' struct as a Python dictionary
    # options = {
    #     'model_type': 'data',
    #     'parameters': {},  # Empty struct in MATLAB
    #     'x0': float('nan'),  # Optional input as NaN
    #     'lb': matlab.double([43.324458513828800]),
    #     'ub': matlab.double([43.324458513828800]),
    # }
    tdc_out, ce_dc = cc_model.dc_model_physical(Tamb_C, Tin_C, qdc_m3h, wdc, nargout=2)
    Tdc_out.append(tdc_out)
    Ce_dc.append(ce_dc)
    
results_df = pd.DataFrame({
    "Tamb": df["Tamb"],
    "Tout": Tdc_out,
    "Ce_dc": Ce_dc,
    "q": df["q"],
    "w_fan": df["w_fan"],
})

results_df


,Tamb,Tout,Ce_dc,q,w_fan
0,22.43,33.380273,71.824243,5.98,13.72
1,18.90,35.344140,32.939683,5.83,11.00
2,15.29,38.086465,194.095312,23.58,25.00
3,15.25,34.086936,135.199550,17.73,19.62
4,27.86,35.756137,113.220947,14.91,17.42
5,13.69,30.279460,236.150353,11.97,27.95
6,25.37,33.011182,889.795006,12.61,47.05
7,15.10,31.775058,32.939683,5.31,11.00
8,19.85,35.724327,180.643986,14.94,23.90
9,13.34,32.384927,378.011529,23.47,34.64


In [ ]:
results_df.to_csv("dc_out_mod_XX.csv")


### Take from pre-evaluated

In [3]:
df = pd.read_csv(data_path / "dc_out_exp.csv")
results_df = pd.read_csv(data_path / "dc_out_mod_physical.csv")

display(df)
display(results_df)


,Tamb,Tin,w_fan,q,Tout
0,22.43,37.54,13.72,5.98,32.95
1,18.90,40.00,11.00,5.83,34.44
2,15.29,42.23,25.00,23.58,38.34
3,15.25,37.55,19.62,17.73,34.00
4,27.86,37.27,17.42,14.91,35.67
5,13.69,37.50,27.95,11.97,30.90
6,25.37,38.90,47.05,12.61,33.06
7,15.10,36.99,11.00,5.31,31.23
8,19.85,40.21,23.90,14.94,36.03
9,13.34,37.43,34.64,23.47,32.85


,Tout
0,33.380273
1,35.344140
2,38.086465
3,34.086936
4,35.756137
5,30.279460
6,33.011182
7,31.775058
8,35.724327
9,32.384927


In [4]:
from phd_visualizations.super_makers import SuperMarker

# fig = regression_plot(
#     df_ref= df, 
#     df_mod= results_df, 
#     var_ids=["Tout"], 
#     units=["℃"]*1, 
#     instruments=[SupportedInstruments.pt100]*1,
#     show_error_metrics=["r2", "mae"],
#     var_labels=["Outlet temperature"],
#     legend_pos="top_spaced",
#     inline_error_metrics_text=True,
    
#     # kwargs for layout
#     title_text="<b>Condenser</b> model validation",
#     legend_y = 1.06, 
#     title_y=0.99,
#     # legend_font=dict(size=12),
#     width=470,
#     height=700,
#     vertical_spacing=0.2,
#     # margin=dict(l=10, r=10, t=200, b=10),
    
#     # super_marker=SuperMarker(
#     #     size_var_id="mv",
#     #     size_var_range=[df.mv.min(), df.mv.max()],
#     #     size_label="Vapor mass flow rate",
#     # )
# )
# fig.update_layout(legend_y = 1.1, legend_font=dict(size=12))

fig = regression_plot(
    df_ref= df, 
    df_mod= results_df, 
    var_ids=["Tout"], 
    units=["℃"], 
    instruments=[SupportedInstruments.pt100],
    width=480, height=750//2,
    show_error_metrics=["r2", "mae"],
    inline_error_metrics_text=True,
    legend_pos="top_spaced",
    vertical_spacing=0.15,
    legend_y=1.15,
    title_y=0.98,
    var_labels=["Outlet temperature"],
    title_text='<span style="color:#83b366; font-weight:bold;">DC</span> model validation'
)

fig


In [ ]:
save_figure(
    fig,
    figure_name="dc_model_regression",
    figure_path=Path("../results"),
    formats=["png", "svg"],
)
